# Lecture 14: Riding the limit: boundary arcs, touch points and costate jumps

**The challenge.** Lecture 13's penalty solution approached the true answer
from below and never reached it. At the weights where it got close, single
shooting was nearly singular. The penalty also blurred the very features that
matter in practice: *when* does the motion reach the limit, how long does it
ride it, and what force does riding it take? For an end stop, a speed limit or
a pressure limit, those are the numbers an engineer needs. They come from the
exact constrained extremal, which has a definite structure: free arcs, where
the limit is inactive, joined to **boundary arcs**, where $g \equiv 0$, or to
isolated **touch points**. At each junction there are conditions, and the
costates can *jump*.

This lecture derives those conditions, solves exactly with the structure
given, checks the result, and explains why order-3 limits are different.

**After this lecture you can**

- derive the control and multiplier on a boundary arc, and the junction and
  jump conditions (indirect adjoining);
- solve exactly with `MultiArcShooting` boundary arcs and touch points,
  seeded from a penalty solution with `from_penalty`;
- read the sign conditions in `diagnostics` to accept or reject a structure;
- combine limits with bang-bang control and singular arcs;
- explain chattering at order 3, and what to do about it.

## 1. Thirty years of getting it right

State constraints resisted a clean theory for a long time. Bryson, Denham
and Dreyfus wrote down conditions in 1963 by differentiating the constraint
until the control appeared and adjoining that derivative to the Hamiltonian.
Others adjoined $g$ itself. The two choices give different multipliers and
different jump conditions, and for years papers disagreed, sometimes
without noticing. David Jacobson, M. M. Lele and Jason Speyer, at Harvard,
showed in 1971 that the structure depends on the order. An order-1 limit
is ridden on boundary arcs but cannot be touched at isolated points; an
order-2 limit can be touched or ridden; and from order 3 on, a smooth entry
onto a boundary arc is in general impossible. Helmut Maurer in Münster
sharpened the junction conditions in 1977. In 1980 H. M. Robbins proved that
an optimal path meeting an order-3 limit generally touches it infinitely
often. In 1995 Richard Hartl, Suresh Sethi and Raymond Vickson published a
survey in *SIAM Review* whose main purpose was to sort out thirty years of
inconsistent statements, and it is still the reference.

The conditions below follow the "indirect adjoining" of Bryson, Denham and
Dreyfus, as made precise by Maurer.

## 2. The conditions

Let $g$ have order $q$, so $g^{(q)}(x, u)$ is the first derivative containing
the control.

**On a boundary arc** $g \equiv 0$, so $g^{(q)} = 0$, and that fixes the
control. Adjoin it with a multiplier $\mu(t)$: minimize
$H + \mu\, g^{(q)}$, so $u_b$ and $\mu$ solve

$$
H_u + \mu\, g^{(q)}_u = 0, \qquad g^{(q)}(x, u_b) = 0,
$$

and the costates follow $\dot\lambda = -\partial(H + \mu g^{(q)})/\partial x$.
A minimizing arc needs $\mu \ge 0$: the limit pushes the right way.

**At the entry** $t_1$ the state must arrive on the limit with all lower
derivatives zero, $N = (g, \dot g, \ldots, g^{(q-1)}) = 0$, a tangential
arrival. The costates may jump:

$$
\lambda(t_1^+) = \lambda(t_1^-) - N_x^\top \pi,
$$

with $q$ unknown multipliers $\pi$. They are continuous at the exit. $H$ is
continuous at both junctions.

**A touch point** (order 2) is a boundary arc of zero length: at the contact
$g = \dot g = 0$, and $\lambda^+ = \lambda^- - \nu\, g_x^\top$ with $\nu \ge 0$.

The package derives the boundary-arc control and multiplier:

In [ ]:
# The three regimes of Bryson-Denham in closed form (Bryson, Denham and Dreyfus 1963):
# the limit x <= l inactive, touched at one instant, or ridden for an interval.
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
t = np.linspace(0, 1, 801)


def closed_form(ell):
    if ell >= 0.25:
        return t * (1 - t), -2 + 0 * t
    s = np.minimum(t, 1 - t)                         # the solution is symmetric about t = 1/2
    if ell >= 1 / 6:
        b, a = 16 * (0.25 - ell), -4 + 12 * ell
        return s + a * s**2 + b * s**3, 2 * a + 6 * b * s
    inside = s < 3 * ell
    x = np.where(inside, ell * (1 - (1 - s / (3 * ell)) ** 3), ell)
    u = np.where(inside, -2 / (3 * ell) * (1 - s / (3 * ell)), 0.0)
    return x, u


cases = [(0.3, "l = 0.3: never reached"), (0.2, "l = 0.2: a touch point at t = 1/2"),
         (0.1, "l = 0.1: a boundary arc on [3l, 1 - 3l]")]
fig, axes = plt.subplots(2, 3, figsize=(12.5, 4.8), sharex=True)
for j, (ell, title) in enumerate(cases):
    x, u = closed_form(ell)
    on = np.abs(x - ell) < 1e-9
    axes[0, j].axhline(ell, color="k", lw=1.2)
    axes[0, j].fill_between(t, ell, 0.32, color=GRAY, alpha=0.15, lw=0)
    axes[0, j].plot(t, x, color=BLUE, lw=2.2)
    axes[0, j].plot(t[on], x[on], color=ORANGE, lw=4)
    if 1 / 6 <= ell < 0.25:
        axes[0, j].plot(0.5, ell, "o", color=ORANGE, ms=8)
    axes[0, j].set(ylim=(0, 0.32))
    axes[0, j].set_title(title, fontsize=10, loc="left")
    axes[1, j].plot(t, u, color=BLUE, lw=2.2)
    axes[1, j].set(xlabel="t", ylim=(-7, 1))
axes[0, 0].set_ylabel("position x (gray: forbidden)")
axes[1, 0].set_ylabel("control u")
axes[1, 1].annotate("a kink: the costate\n" r"$\lambda_x$ jumps here", (0.5, closed_form(0.2)[1][400]),
                    xytext=(0.55, -5.5), fontsize=8.5, arrowprops={"arrowstyle": "-|>", "lw": 0.8})
axes[1, 2].annotate("u = 0 on the arc", (0.5, 0), xytext=(0.3, -3), fontsize=8.5,
                    arrowprops={"arrowstyle": "-|>", "lw": 0.8})
for a in axes.ravel():
    a.grid(alpha=0.25)
fig.tight_layout()

*The three regimes of Bryson–Denham, in closed form. The orange dot is a touch point; the thick orange line is a boundary arc, ridden with $u = 0$. At the touch point the control has a kink, because $\lambda_x$ jumps there.*

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from auto_optimal_control import examples
from auto_optimal_control.build import build_kernels
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.shooting import MultiArcShooting, SingleShooting, parameter_continuation

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

bd = derive(examples.bryson_denham())
c = bd.path_constraints[0]
print("order", c.order, "  boundary control:", c.boundary.control,
      "  multiplier mu =", c.boundary.multiplier)
print("costate dynamics on the arc:", c.boundary.dynamics[2:])
k = build_kernels(bd)
print("exact arcs generated per constraint:", k.EXACT_ARCS, " orders:", k.CONSTRAINT_ORDERS)

For Bryson–Denham, riding $x = \ell$ needs $u = 0$ (from $\ddot g = u = 0$),
and $\mu = -\lambda_v$.

## 3. Exact shooting with the structure given

`MultiArcShooting(kernels, params, arcs)` solves the boundary value problem
for a given sequence of special arcs. Its unknowns are single shooting's
plus, per boundary arc, the entry and exit times and the $q$ jump
multipliers $\pi$ (per touch point: the time and $\nu$). Its residuals add
the entry conditions $N = 0$ and $H$ continuity. The Jacobian is exact: the
STM chained across the arcs, with the derivatives of every jump and every
junction time. On free arcs the penalty is off: the limit is not enforced
there, only checked.

Where do the unknown junction times come from? From a penalty solution:
`from_penalty` reads the arcs off its crossings (each arc spanning its first
to last crossing, past the dips Lecture 13 saw), takes its costates, and fits
$\pi$ by integrating the boundary arc back from its exit.

In [ ]:
ell = 0.1
free = SingleShooting(k, [ell, 0.0]).solve([0.0, 2.0, 0.0, 0.0])
rho, pen = parameter_continuation(k, [ell, 1.0], "rho", np.logspace(0, 4, 9), free.v)[-1]
shoot, v0 = MultiArcShooting.from_penalty(k, [ell, 0.0], [("boundary", "x_max")], pen, [ell, rho])
print(shoot)
print("seed from the penalty solution at rho = %.0e:" % rho, shoot.unpack_arcs(v0))
sol = shoot.solve(v0)
(t_in, t_out, pi), = shoot.unpack_arcs(sol.v)
print(f"|R| = {sol.residual_norm:.1e}; boundary arc [{t_in:.12f}, {t_out:.12f}] "
      f"(exact [{3 * ell}, {1 - 3 * ell}])")
print(f"J = {shoot.cost(sol.v):.14f}  (exact 4/(9 l) = {4 / (9 * ell):.14f}); pi = {pi}")

A penalty weight of only $10^4$ was enough to read the structure, and the
exact solve lands on the closed form: the cost to round-off, the junction
times to about $10^{-9}$. (The control is continuous across the entry
here, so $H$ continuity holds to second order in the junction time: a double
root, where Newton converges only linearly. The cost is insensitive to that
error.)
Lecture 13 needed $\rho = 10^7$ for three digits of the cost.

`diagnostics` checks what the residual cannot: the sign of $\mu$ along the
arc, the control there, and violations on the free arcs:

In [ ]:
d = shoot.diagnostics(sol.v)
print("mu on the arc, min and max:", d["boundary_mu"][0])
print("violation on the free arcs:", d["free_arc_violation"])
tr = shoot.trajectory(sol.v, n_points=600)
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(12, 3.2))
a1.plot(tr["t"], tr["z"][:, 0], color=BLUE, lw=2)
a1.axhline(ell, color="k", lw=0.8)
a1.set_title("position x", fontsize=10, loc="left")
a2.plot(tr["t"], tr["u"][:, 0], color=ORANGE, lw=2)
a2.set_title("control u", fontsize=10, loc="left")
a3.plot(tr["t"], tr["z"][:, 3], color=BLUE, lw=2)
a3.set_title(r"costate $\lambda_v$: jumps at the entry", fontsize=10, loc="left")
for a_ in (a1, a2, a3):
    for tj in (t_in, t_out):
        a_.axvline(tj, color=GRAY, lw=0.8, ls=":")
    a_.set_xlabel("t")
    a_.grid(alpha=0.3)
fig.tight_layout()

### The touch regime, and a wrong structure

For $\ell = 0.2$ the exact solution touches the wall once. Ask for a
boundary arc anyway, then for a touch point, and compare what the solver and
the diagnostics say:

In [ ]:
ell2 = 0.2
free2 = SingleShooting(k, [ell2, 0.0]).solve([0.0, 2.0, 0.0, 0.0])
rho2, pen2 = parameter_continuation(k, [ell2, 1.0], "rho", np.logspace(0, 6, 13), free2.v)[-1]
for structure in ([("boundary", "x_max")], [("touch", "x_max")]):
    try:
        sh, g0 = MultiArcShooting.from_penalty(k, [ell2, 0.0], structure, pen2, [ell2, rho2])
        s2 = sh.solve(g0)
        dd = sh.diagnostics(s2.v)
        print(f"{structure[0][0]:8s}: converged {s2.success}, |R| = {s2.residual_norm:.1e}, "
              f"J = {sh.cost(s2.v):.10f}, arcs {sh.unpack_arcs(s2.v)}")
        print(f"          mu range {dd['boundary_mu']}, nu {dd['touch_nu']}")
    except Exception as err:
        print(f"{structure[0][0]:8s}: {type(err).__name__}: {err}")
a, b = -4 + 12 * ell2, 4 - 16 * ell2
print("closed form J =", (2 * a) ** 2 * 0.5 + (2 * a) * (6 * b) * 0.25 + (6 * b) ** 2 * 0.125 / 3)

The touch point reproduces the closed form, with $\nu > 0$ as required. The
wrong structure does not survive. Reading these outcomes (a failed solve,
an arc of negative length, a multiplier of the wrong sign, a violation on a
free arc) is how a structure is accepted or rejected. Lecture 15 automates
exactly this.

## 4. Limits with bounded and bang-bang controls

On a free arc a bounded control keeps its usual law, clipped or bang-bang,
with its switches located as always. On a boundary arc the control is the
one that rides the limit, and it must stay within its bounds:
`diagnostics` reports `boundary_bound_violation`.

**A clipped control.** The package's `velocity_limited_move` with a force
limit: full force, a ramp onto the speed limit, cruise, and the mirror
image. The closed form is in `examples.velocity_limited_move_bounded_exact`.

In [ ]:
kv = build_kernels(derive(examples.velocity_limited_move(bounded=True)))
V, umax = 1.2, 8.0
fv = SingleShooting(kv, [V, 1.0, umax]).solve([12.0, 6.0, 0, 0])
rv, penv = parameter_continuation(kv, [V, 1.0, umax], "rho", np.logspace(0, 2, 9), fv.v)[-1]
sv, gv = MultiArcShooting.from_penalty(kv, [V, 0.0, umax], [("boundary", "v_max")], penv,
                                       [V, rv, umax])
solv = sv.solve(gv)
J_exact, t1, s_full = examples.velocity_limited_move_bounded_exact(V, umax)
print(f"J = {sv.cost(solv.v):.12f} (closed form {J_exact:.12f}); "
      f"entry {sv.unpack_arcs(solv.v)[0][0]:.8f} (closed form {t1:.8f})")
print("control on the arc within its bounds:", sv.diagnostics(solv.v)["boundary_bound_violation"])

**A bang-bang control.** With a minimum-time cost the force is bang-bang,
and a penalty is useless: the penalized optimum rides the limit on a
*singular* arc (Lecture 12), not at a bound. The `Homotopy` of Lecture 15
handles it by smoothing the control and penalizing the limit together, then
reading the structure. The train between two stations, `speed_limited_min_time`,
runs at full power to the speed limit, cruises on it, then brakes:
$t_f = D/V + V/u_{\max}$.

In [ ]:
from auto_optimal_control.homotopy import Homotopy

V, D, umax = 2.0, 10.0, 0.5
rt = Homotopy(examples.speed_limited_min_time()).solve([V, D, umax], ([-3, -3], [0, 3]), (1.0, 20.0))
print(rt.message, rt.arcs)
print(f"t_f = {rt.solution.tf:.14f} (closed form {D / V + V / umax})")

**A singular arc joined to a boundary arc.** Goddard's rocket of Lecture 12,
with a limit on dynamic pressure $q = \tfrac12\rho_0 v^2 e^{-\beta(h-1)} \le 10$,
an order-1 limit. Unconstrained, $q$ peaks at about 15 right where the
singular arc begins. With the limit, the rocket rides $q = 10$, then
*joins* the singular arc directly with no free arc between: the entry
`("singular", "u", "upper", "lower", "joined")`, whose conditions
$S = \dot S = 0$ at the junction replace the boundary arc's $H$ continuity at
its exit.

In [ ]:
rq = Homotopy(examples.goddard(dynamic_pressure=True)).solve(
    examples.GODDARD_Q_PARAMS, ([-6, -0.5, -0.1], [-1, 0, 0.1]), (0.15, 0.3))
print(rq.message)
print("structure:", rq.arcs)
print(f"h(t_f) = {-rq.objective:.9f} (unconstrained 1.012836918), "
      f"max q - q_max = {rq.solution.max_violation:.1e}")
dq = rq.shooting.diagnostics(rq.solution.v)
print("mu on the boundary arc:", dq["boundary_mu"][0], " Legendre-Clebsch:", dq["singular_legendre"][0])

## 5. Order three: chattering, and what to do

For an order-3 limit, such as an end stop on a jerk-controlled axis, Robbins
showed that an optimal path generally cannot enter a boundary arc
smoothly. Instead it touches the limit at a sequence of points that pile up
infinitely fast towards the entry: **chattering**. No finite structure
represents that, so the package does not generate exact arcs at order 3 or
more, and the automatic solve says so:

In [ ]:
# A sketch of chattering at an order-3 limit: instead of entering the boundary arc, the
# path touches the limit at instants that pile up geometrically towards the entry time.
import matplotlib.pyplot as plt
import numpy as np
import itertools

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
t_entry, q = 1.0, 0.45
touches = t_entry - 0.6 * q ** np.arange(0, 14)
fig, ax = plt.subplots(figsize=(8, 3.2))
ts, xs = [np.linspace(0, touches[0], 100)], [1 - (1 - np.linspace(0, touches[0], 100) / touches[0]) ** 3]
for a_, b_ in itertools.pairwise(touches):
    s = np.linspace(0, 1, 60)
    ts.append(a_ + (b_ - a_) * s)
    xs.append(1 - 6 * (b_ - a_) ** 3 * (16 * (s * (1 - s)) ** 2))
ts.append(np.linspace(touches[-1], 1.4, 50))
xs.append(np.ones(50))
ax.plot(np.concatenate(ts), np.concatenate(xs), color=BLUE, lw=2)
ax.plot(touches, np.ones_like(touches), "o", color=ORANGE, ms=5, zorder=5)
ax.axhline(1.0, color="k", lw=1)
ax.axvline(t_entry, color=GRAY, lw=0.8, ls=":")
ax.text(t_entry + 0.01, 0.7, "the boundary arc\nwould start here", fontsize=9, color="k")
ax.text(0.45, 1.02, "touch points, infinitely many before the entry", fontsize=9, color=ORANGE,
        va="bottom")
ax.set(xlabel="t", ylabel="constrained quantity", ylim=(0.4, 1.12), xlim=(0, 1.4))
ax.set_yticks([1.0], ["limit"])
ax.set_title("order three: chattering (a sketch, not to scale)", fontsize=10, loc="left")
ax.grid(alpha=0.25)
fig.tight_layout()

*Chattering, sketched. Instead of one entry onto the limit, the path touches it infinitely often, at times that pile up geometrically. No finite list of arcs describes it.*

In [ ]:
from auto_optimal_control.problem import OptimalControlProblem, PathConstraint

t, x, v, a, u = sp.symbols("t x v a u", real=True)
jerk_stop = OptimalControlProblem(
    name="jerk_end_stop", time=t, states=[x, v, a], controls=[u], params=[sp.Symbol("l")],
    dynamics=[v, a, u], running_cost=u**2 / 2, initial_state=[0, 1, 0],
    final_state={x: 0, v: -1, a: 0}, final_time=1,
    path_constraints=[PathConstraint(x - sp.Symbol("l"), "x_max")])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    r3 = Homotopy(jerk_stop).solve([0.1], ([-50] * 3, [50] * 3), max_steps=40)
print("exact:", r3.exact, "-", r3.message)
print(f"best penalty solution: max violation {r3.solution.max_violation:.1e} at rho = {r3.params[-1]:.2e}")

What to do in practice, in order of preference:

1. **Check whether the limit is active at all.** Often it is not, and
   `terminate` (Lecture 13) is all that is needed: reject trajectories that
   violate it.
2. **Reformulate to a lower order.** For an end stop, constrain "can still
   stop before the end", $x + d_{\text{stop}}(v, a) - x_{\max} \le 0$, where
   $d_{\text{stop}}$ is the braking distance under the jerk limit. That
   depends on $a$, so it is order 1, and it is what motion planners use.
3. **Penalty with a margin.** Approximate, but genuinely feasible if the limit
   is tightened by the remaining violation.

The other current limits of exact arcs, listed in the README: two
constraints active at once, one limit with several bang-bang controls,
limits that depend explicitly on time, and a singular arc that ends directly
on a boundary arc.

## Summary

- On a boundary arc of an order-$q$ limit, $g^{(q)} = 0$ fixes the control;
  $\mu$ from $H_u + \mu g^{(q)}_u = 0$ must be $\ge 0$. At the entry
  $N = (g, \ldots, g^{(q-1)}) = 0$ and $\lambda$ jumps by $-N_x^\top\pi$; $H$ is
  continuous at both junctions. Touch points (order 2) jump by
  $-\nu g_x^\top$ with $\nu \ge 0$.
- `MultiArcShooting` solves a given structure exactly, with an exact
  Jacobian; `from_penalty` seeds it from a penalty solution; `diagnostics`
  (`boundary_mu`, `touch_nu`, `free_arc_violation`,
  `boundary_bound_violation`) accepts or rejects it.
- Bounded and bang-bang controls keep their laws on free arcs; singular arcs
  can join boundary arcs (`"joined"`).
- Order-3 limits chatter; exact arcs stop at order 2.

## Exercises

1. Solve Bryson–Denham exactly for $\ell$ from 0.05 to 0.24, choosing the
   structure from the closed form, and plot the jump multipliers $\pi$ and
   $\nu$ against $\ell$. What happens near $\ell = 1/6$?
2. For the velocity limit without a force bound
   (`velocity_limited_move()`), find the smallest $V$ for which the problem
   is feasible and solve near it. What happens to $\mu$?
3. Implement the braking-distance reformulation of the jerk end stop as an
   order-1 `PathConstraint` and solve it with `Homotopy`. Compare with the
   penalty solution of the order-3 limit.

## Further reading

- R. F. Hartl, S. P. Sethi and R. G. Vickson, "A survey of the maximum
  principles for optimal control problems with state constraints",
  *SIAM Review* 37(2), 1995.
- D. H. Jacobson, M. M. Lele and J. L. Speyer, "New necessary conditions of
  optimality for control problems with state-variable inequality
  constraints", *Journal of Mathematical Analysis and Applications* 35, 1971.
- H. Maurer, "On optimal control problems with bounded state variables and
  control appearing linearly", *SIAM Journal on Control and Optimization*
  15, 1977.
- H. M. Robbins, "Junction phenomena for optimal control with state-variable
  inequality constraints of third order", *Journal of Optimization Theory
  and Applications* 31, 1980.